# Notebook 2: Clustering and Regime Analysis

This notebook covers the second half of the dual-axis pipeline:
1. Diffusion Maps (dimensionality reduction)
2. Spectral gap analysis
3. ToMATo clustering
4. Separation diagnostic (eta-squared)
5. Tukey HSD merge
6. Regime characterization (economic and dynamic axes)
7. Quasi-orthogonality test (ARI)
8. Mean-reversion analysis
9. BIC model comparison

It loads pre-computed features and embeddings from **Notebook 1**.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import warnings
from pathlib import Path
from scipy.stats import studentized_range, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

ROOT = Path('..').resolve()
DATA = ROOT / 'results_darcsinh' / 'split_W512_S6'

SEED = 42
W, S = 512, 6
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Load Data

We load the pre-computed outputs from Notebook 1:
- `preprocessed.parquet`: hourly time series (datetime, lmp, r, dr)
- `fe_features.parquet`: 15 financial-engineering features per window
- `moment_embeddings.parquet`: 1024-dimensional MOMENT embeddings per window

In [ ]:
# Load pre-computed data from Notebook 1
df_pre = pd.read_parquet(DATA / 'preprocessed.parquet')
df_fe = pd.read_parquet(DATA / 'fe_features.parquet')
df_mom = pd.read_parquet(DATA / 'moment_embeddings.parquet')

FE_NAMES = ['mean', 'std', 'skew', 'kurt', 'min', 'max', 'range',
            'median', 'p5', 'p95', 'iqr', 'vol_24h',
            'lmp_mean', 'lmp_p95', 'lmp_std']

fe = df_fe[FE_NAMES].values.astype(np.float32)
mom_cols = [c for c in df_mom.columns if c.startswith('mom_')]
mom = df_mom[mom_cols].values.astype(np.float32)
ts = pd.to_datetime(df_fe['datetime'].values)
N = len(fe)

print(f'FE features:       {fe.shape}')
print(f'MOMENT embeddings: {mom.shape}')
print(f'Windows: N = {N:,}')

In [ ]:
# Reconstruct window arrays for merge targets and diagnostics
r = df_pre['r'].values
dr = df_pre['dr'].values
lmp = df_pre['lmp'].values
dt = df_pre['datetime'].values

def make_windows(vals, lmp_arr, dt_arr):
    starts = list(range(0, len(vals) - W + 1, S))
    wv = np.array([vals[s:s+W] for s in starts], dtype=np.float32)
    wl = np.array([lmp_arr[s:s+W] for s in starts], dtype=np.float32)
    return wv, wl

wr_mom, wl = make_windows(r, lmp, dt)
print(f'Reconstructed window arrays: {wr_mom.shape}')

### Helper Functions

Autocorrelation and eta-squared utilities used throughout the notebook.

In [ ]:
def _acf(x, lag):
    """Autocorrelation at a given lag."""
    n = len(x); m = x.mean(); v = ((x - m)**2).sum()
    if v < 1e-15 or lag >= n: return 0.0
    return float(((x[:n-lag] - m) * (x[lag:] - m)).sum() / v)


def eta2(vals, labels):
    """Correlation ratio: fraction of variance explained by grouping."""
    mask = labels >= 0
    x, z = vals[mask], labels[mask]
    gm = x.mean(); ss_t = ((x - gm)**2).sum()
    if ss_t < 1e-15: return 0.0
    ss_b = sum(len(x[z == k]) * (x[z == k].mean() - gm)**2 for k in np.unique(z))
    return float(ss_b / ss_t)

## 2. Diffusion Maps

Diffusion Maps embed each window in a low-dimensional space that preserves the geometry of the data manifold. A Gaussian kernel defines transition probabilities between windows, and the leading eigenvectors of the transition matrix provide the diffusion coordinates.

The dimensionality d is determined by the spectral gap: a sharp drop in eigenvalue magnitude signals the transition from signal to noise.

In [ ]:
def diffusion_maps(X, label='', fixed_d=None):
    """Compute Diffusion Maps embedding.
    
    Parameters
    ----------
    X : array (N, p)
        Input features (will be standardized internally).
    label : str
        Label for print output.
    fixed_d : int or None
        If given, use this dimensionality. Otherwise select d by
        silhouette score on KMeans over candidate dimensions.
    
    Returns
    -------
    coords : array (N, d)
        Diffusion coordinates.
    d : int
        Selected dimensionality.
    all_evals : array (20,)
        Top 20 non-trivial eigenvalues.
    """
    Xs = StandardScaler().fit_transform(X)
    if Xs.shape[1] > 50:
        Xs = PCA(n_components=50, random_state=SEED).fit_transform(Xs)

    Xt = torch.tensor(Xs, dtype=torch.float64, device=DEVICE)
    dists = torch.cdist(Xt, Xt)
    eps = float(torch.median(dists[dists > 0]).item()) ** 2
    K = torch.exp(-dists ** 2 / eps)
    P = torch.diag(1.0 / K.sum(dim=1)) @ K
    evals, evecs = torch.linalg.eigh(P)
    evals, evecs = evals.flip(0), evecs.flip(1)
    all_evals = evals[1:21].cpu().numpy()
    all_coords = (evecs[:, 1:21] * evals[1:21]).cpu().numpy()
    del Xt, dists, K, P
    torch.cuda.empty_cache()

    if fixed_d is not None:
        d = fixed_d
    else:
        best_d, best_s = 2, -1
        for d_cand in range(2, 21):
            cd = all_coords[:, :d_cand]
            nc = max(2, min(10, len(cd) // 50))
            km = KMeans(n_clusters=nc, n_init=5, random_state=SEED).fit(cd)
            s = silhouette_score(cd, km.labels_)
            if s > best_s:
                best_d, best_s = d_cand, s
        d = best_d

    coords = all_coords[:, :d]
    print(f'{label}: d={d}, top eigenvalues: {all_evals[:d].round(4)}')
    return coords, d, all_evals


dm_fe, d_fe, evals_fe = diffusion_maps(StandardScaler().fit_transform(fe), 'FE')
dm_mom, d_mom, evals_mom = diffusion_maps(mom, 'MOMENT', fixed_d=5)

## 3. Spectral Gap

The spectral gap identifies the intrinsic dimensionality of each embedding. A clear drop after eigenvalue d means that the first d diffusion coordinates capture the dominant structure, and the remaining coordinates are noise.

- **FE**: gap after eigenvalue 2 (d = 2)
- **MOMENT**: plateau across eigenvalues 2-4, gap after eigenvalue 4 (d = 5, including the residual eigenvalue 5)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, len(evals_fe)+1), evals_fe, color='steelblue', alpha=0.8)
axes[0].axvline(d_fe + 0.5, color='red', ls='--', lw=1.5, label=f'd = {d_fe}')
axes[0].set_xlabel('Eigenvalue index')
axes[0].set_ylabel('Eigenvalue')
axes[0].set_title('FE \u2014 Spectral Gap')
axes[0].legend(frameon=False)

axes[1].bar(range(1, len(evals_mom)+1), evals_mom, color='darkorange', alpha=0.8)
axes[1].axvline(d_mom + 0.5, color='red', ls='--', lw=1.5, label=f'd = {d_mom}')
axes[1].set_xlabel('Eigenvalue index')
axes[1].set_title('MOMENT \u2014 Spectral Gap')
axes[1].legend(frameon=False)

plt.tight_layout()
plt.show()

### Diffusion Coordinate Scatter

Visualize the first two diffusion coordinates for each branch, colored by the respective target variable (LMP mean for FE, ACF at lag 6h for MOMENT).

In [ ]:
# Pre-compute target variables
lmp_mean = np.array([wl[i].mean() for i in range(N)])
acf6 = np.array([_acf(wr_mom[i].astype(np.float64), 6) for i in range(N)])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc0 = axes[0].scatter(dm_fe[:, 0], dm_fe[:, 1], c=lmp_mean,
                       cmap='RdYlBu_r', s=2, alpha=0.5)
axes[0].set_xlabel('\u03a8\u2081'); axes[0].set_ylabel('\u03a8\u2082')
axes[0].set_title('FE Diffusion Coordinates (colored by LMP)')
plt.colorbar(sc0, ax=axes[0], label='LMP mean ($/MWh)')

sc1 = axes[1].scatter(dm_mom[:, 0], dm_mom[:, 1], c=acf6,
                       cmap='RdYlBu_r', s=2, alpha=0.5)
axes[1].set_xlabel('\u03a8\u2081'); axes[1].set_ylabel('\u03a8\u2082')
axes[1].set_title('MOMENT Diffusion Coordinates (colored by ACF 6h)')
plt.colorbar(sc1, ax=axes[1], label='ACF 6h')

plt.tight_layout()
plt.show()

## 4. ToMATo Clustering

ToMATo (Topological Mode Analysis Tool) identifies density modes using k-NN density estimation and persistent homology. The number of clusters K emerges from the data — no a priori specification needed.

The persistence diagram determines how many modes survive above the noise threshold.

In [ ]:
from gudhi.clustering.tomato import Tomato

def tomato_cluster(X):
    """Run ToMATo over a grid of k-NN values and select the configuration
    that discovers the most modes."""
    best_lab, best_k, best_n = None, None, 0
    for k in [20, 40, 60, 80, 100, 150]:
        tmt = Tomato(density_type='KDE', graph_type='knn', n_neighbors=k)
        tmt.fit(X)
        if hasattr(tmt, 'diagram_') and len(tmt.diagram_) > 1:
            deaths = np.sort([d for _, d in tmt.diagram_ if d < np.inf])
            if len(deaths) > 1:
                n = len(deaths) - np.argmax(np.diff(deaths))
                tmt.n_clusters_ = n
            else:
                n = 1
        else:
            n = 1
        if n > best_n:
            best_n, best_lab, best_k = n, tmt.labels_.copy(), k
    return best_lab, best_k, best_n


lab_fe, knn_fe, modes_fe = tomato_cluster(dm_fe)
lab_mom, knn_mom, modes_mom = tomato_cluster(dm_mom)
print(f'FE:     {modes_fe} modes (k-NN: k={knn_fe})')
print(f'MOMENT: {modes_mom} modes (k-NN: k={knn_mom})')

## 5. Separation Diagnostic (\u03b7\u00b2)

The correlation ratio \u03b7\u00b2 measures the fraction of a feature's variance explained by the clustering. For each of 19 features (15 FE features + 4 diagnostic ACFs), we compute \u03b7\u00b2 against both partitions.

The result reveals which features each branch separates: FE separates price features, MOMENT separates persistence features. The merge variable for each branch is the feature with the highest \u03b7\u00b2.

In [ ]:
rows = []
for j, name in enumerate(FE_NAMES):
    rows.append({'feature': name, 'group': 'FE',
                 'eta2_FE': eta2(fe[:, j], lab_fe),
                 'eta2_MOM': eta2(fe[:, j], lab_mom)})

for lag, name in [(1, 'acf_1h'), (6, 'acf_6h'), (24, 'acf_24h'), (168, 'acf_168h')]:
    vals = np.array([_acf(wr_mom[i].astype(np.float64), lag) for i in range(N)])
    rows.append({'feature': name, 'group': 'ACF diagnostic',
                 'eta2_FE': eta2(vals, lab_fe),
                 'eta2_MOM': eta2(vals, lab_mom)})

df_eta = pd.DataFrame(rows)
print(df_eta.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = {'FE': 'steelblue', 'ACF diagnostic': 'darkorange'}
markers = {'FE': 'o', 'ACF diagnostic': '^'}
for grp in df_eta['group'].unique():
    mask = df_eta['group'] == grp
    ax.scatter(df_eta.loc[mask, 'eta2_FE'], df_eta.loc[mask, 'eta2_MOM'],
               c=colors[grp], marker=markers[grp], s=60, label=grp,
               alpha=0.8, edgecolors='white', lw=0.5)
    for _, row in df_eta[mask].iterrows():
        ax.annotate(row['feature'],
                    (row['eta2_FE']+0.008, row['eta2_MOM']+0.008), fontsize=7)

ax.set_xlabel('\u03b7\u00b2 (FE partition)')
ax.set_ylabel('\u03b7\u00b2 (MOMENT partition)')
ax.set_title('Separation Diagnostic')
ax.plot([0, 0.55], [0, 0.55], 'k--', lw=0.5, alpha=0.3)
ax.legend(frameon=False)
ax.set_xlim(-0.02, 0.55)
ax.set_ylim(-0.02, 0.55)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print(f'\nMerge variables:')
print(f'  FE:     lmp_mean (\u03b7\u00b2_FE = {eta2(lmp_mean, lab_fe):.3f})')
print(f'  MOMENT: acf_6h   (\u03b7\u00b2_MOM = {eta2(acf6, lab_mom):.3f})')

## 6. Tukey HSD Merge

The ToMATo modes are fine-grained (48 for FE, 47 for MOMENT). Many adjacent modes are not statistically distinguishable on the merge variable. Tukey HSD iteratively merges the closest pair whose difference is not significant (\u03b1 = 0.05), collapsing both partitions to 9 regimes.

In [ ]:
def tukey_merge(labels, target, alpha=0.05, min_size=20):
    """Iteratively merge ToMATo modes using Tukey HSD on a target variable.
    
    First, tiny clusters (< min_size) are absorbed into their nearest
    neighbor by target-variable mean. Then, pairs of clusters whose
    means are not significantly different (Tukey HSD at level alpha)
    are merged, smallest difference first, until all remaining pairs
    are statistically distinguishable.
    """
    m = labels.copy()

    def _relabel(m):
        for j, v in enumerate(np.unique(m[m >= 0])):
            m[m == v] = j
        return m

    # Absorb tiny clusters
    u, counts = np.unique(m[m >= 0], return_counts=True)
    for k, c in sorted(zip(u, counts), key=lambda x: x[1]):
        if c < min_size and len(np.unique(m[m >= 0])) > 1:
            mk = target[m == k].mean()
            others = [kk for kk in np.unique(m[m >= 0]) if kk != k]
            nearest = min(others, key=lambda kk: abs(target[m == kk].mean() - mk))
            m[m == k] = nearest
    m = _relabel(m)

    # Iterative Tukey HSD merge
    while True:
        mask = m >= 0
        x, z = target[mask], m[mask]
        u = np.unique(z)
        K = len(u)
        if K <= 1:
            break
        N_total = len(x)
        g_means = np.array([x[z == k].mean() for k in u])
        g_ns = np.array([np.sum(z == k) for k in u])
        ss_within = sum(((x[z == k] - x[z == k].mean())**2).sum() for k in u)
        df_within = N_total - K
        if df_within <= 0:
            break
        mse = ss_within / df_within
        order = np.argsort(g_means)
        g_means_s, g_ns_s, u_s = g_means[order], g_ns[order], u[order]
        q_crit = studentized_range.ppf(1 - alpha, K, df_within)
        best_pair, best_diff = None, np.inf
        for i in range(K):
            for j in range(i + 1, K):
                diff = abs(g_means_s[i] - g_means_s[j])
                se = np.sqrt(mse * 0.5 * (1.0/g_ns_s[i] + 1.0/g_ns_s[j]))
                q_stat = diff / se if se > 1e-15 else np.inf
                if q_stat < q_crit and diff < best_diff:
                    best_diff = diff
                    best_pair = (u_s[i], u_s[j])
        if best_pair is None:
            break
        m[m == best_pair[1]] = best_pair[0]
        m = _relabel(m)

    return m


lab_fe_m = tukey_merge(lab_fe, lmp_mean)
lab_mom_m = tukey_merge(lab_mom, acf6)
K_fe = len(np.unique(lab_fe_m[lab_fe_m >= 0]))
K_mom = len(np.unique(lab_mom_m[lab_mom_m >= 0]))
print(f'Economic regimes: {modes_fe} modes \u2192 {K_fe} regimes')
print(f'Dynamic regimes:  {modes_mom} modes \u2192 {K_mom} regimes')

## 7. Regime Characterization

### 7a. Economic Axis

The 9 economic regimes span from off-peak ($30/MWh) to extreme winter spike ($151/MWh). The dominant regime is E3 (Baseload, 35% of windows).

In [ ]:
rows_e = []
for e in np.sort(np.unique(lab_fe_m[lab_fe_m >= 0])):
    mask = lab_fe_m == e
    rows_e.append({
        'Regime': f'E{e}', 'LMP mean': round(lmp_mean[mask].mean(), 1),
        'LMP std': round(lmp_mean[mask].std(), 1),
        'n': int(mask.sum()), '%': round(100 * mask.sum() / N, 1)
    })
df_E = pd.DataFrame(rows_e).sort_values('LMP mean')
print('Economic Regimes:')
print(df_E.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
order_e = df_E['Regime'].values
data_e = [lmp_mean[lab_fe_m == int(reg[1:])] for reg in order_e]
bp = ax.boxplot(data_e, labels=order_e, patch_artist=True, widths=0.6)
for patch in bp['boxes']:
    patch.set_facecolor('steelblue')
    patch.set_alpha(0.6)
ax.set_ylabel('LMP Mean ($/MWh)')
ax.set_title('Economic Regimes \u2014 Price Distribution')
ax.grid(alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

### 7b. Dynamic Axis

The 9 dynamic regimes capture the persistence structure of the residual process. Low-ACF regimes correspond to fast mean-reversion (shocks decay in hours), while high-ACF regimes correspond to slow mean-reversion (shocks persist for days).

In [ ]:
rows_d = []
for d in np.sort(np.unique(lab_mom_m[lab_mom_m >= 0])):
    mask = lab_mom_m == d
    rows_d.append({
        'Regime': f'D{d}', 'ACF 6h': round(acf6[mask].mean(), 3),
        'sigma_r': round(np.array([wr_mom[i].std() for i in range(N)])[mask].mean(), 3),
        'LMP mean': round(lmp_mean[mask].mean(), 1),
        'n': int(mask.sum()), '%': round(100 * mask.sum() / N, 1)
    })
df_D = pd.DataFrame(rows_d).sort_values('ACF 6h')
print('Dynamic Regimes:')
print(df_D.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
order_d = df_D['Regime'].values
data_d = [acf6[lab_mom_m == int(reg[1:])] for reg in order_d]
bp = ax.boxplot(data_d, labels=order_d, patch_artist=True, widths=0.6)
for patch in bp['boxes']:
    patch.set_facecolor('darkorange')
    patch.set_alpha(0.6)
ax.set_ylabel('ACF at lag 6h')
ax.set_title('Dynamic Regimes \u2014 Persistence Distribution')
ax.grid(alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

## 8. Quasi-Orthogonality

If the two axes capture the same structure, their partitions should agree (ARI \u2248 1). If they capture independent dimensions, ARI \u2248 0.

In [ ]:
ari = adjusted_rand_score(lab_fe_m, lab_mom_m)
print(f'ARI between economic and dynamic partitions: {ari:.3f}')
print(f'\nARI \u2248 0 \u2192 the two axes are quasi-orthogonal.')

In [ ]:
# Cross-tabulation heatmap
valid = (lab_fe_m >= 0) & (lab_mom_m >= 0)
ct = pd.crosstab(lab_fe_m[valid], lab_mom_m[valid])
ct.index = [f'E{i}' for i in ct.index]
ct.columns = [f'D{j}' for j in ct.columns]
populated = (ct > 0).sum().sum()
print(f'Populated cells: {populated} / {ct.shape[0] * ct.shape[1]}')

fig, ax = plt.subplots(figsize=(8, 6))
grid = ct.values.astype(float)
grid[grid == 0] = np.nan
im = ax.imshow(grid, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(grid.shape[1]))
ax.set_xticklabels(ct.columns)
ax.set_yticks(range(grid.shape[0]))
ax.set_yticklabels(ct.index)
ax.set_xlabel('Dynamic regime')
ax.set_ylabel('Economic regime')
ax.set_title(f'Two-Axis Regime Grid (ARI = {ari:.3f})')
plt.colorbar(im, label='Window count')
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        if not np.isnan(grid[i, j]):
            ax.text(j, i, f'{int(grid[i,j])}', ha='center', va='center', fontsize=7)
plt.tight_layout()
plt.show()

### Axes Evidence

A 2x2 panel confirms the separation: FE regimes separate price but not persistence, while MOMENT regimes separate persistence but not price.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sort regimes by their target variable
e_order = np.argsort([lmp_mean[lab_fe_m == e].mean() for e in range(K_fe)])
d_order = np.argsort([acf6[lab_mom_m == d].mean() for d in range(K_mom)])

# (a) FE regimes vs LMP
for idx, e in enumerate(e_order):
    vals = lmp_mean[lab_fe_m == e]
    bp = axes[0, 0].boxplot([vals], positions=[idx], widths=0.6, patch_artist=True)
    bp['boxes'][0].set_facecolor('steelblue'); bp['boxes'][0].set_alpha(0.6)
axes[0, 0].set_xticks(range(K_fe))
axes[0, 0].set_xticklabels([f'E{e}' for e in e_order])
axes[0, 0].set_ylabel('LMP Mean ($/MWh)')
axes[0, 0].set_title('(a) FE regimes vs Price \u2014 clear separation')

# (b) MOMENT regimes vs LMP
for idx, d in enumerate(d_order):
    vals = lmp_mean[lab_mom_m == d]
    bp = axes[0, 1].boxplot([vals], positions=[idx], widths=0.6, patch_artist=True)
    bp['boxes'][0].set_facecolor('darkorange'); bp['boxes'][0].set_alpha(0.6)
axes[0, 1].set_xticks(range(K_mom))
axes[0, 1].set_xticklabels([f'D{d}' for d in d_order])
axes[0, 1].set_ylabel('LMP Mean ($/MWh)')
axes[0, 1].set_title('(b) MOMENT regimes vs Price \u2014 no separation')

# (c) MOMENT regimes vs ACF
for idx, d in enumerate(d_order):
    vals = acf6[lab_mom_m == d]
    bp = axes[1, 0].boxplot([vals], positions=[idx], widths=0.6, patch_artist=True)
    bp['boxes'][0].set_facecolor('darkorange'); bp['boxes'][0].set_alpha(0.6)
axes[1, 0].set_xticks(range(K_mom))
axes[1, 0].set_xticklabels([f'D{d}' for d in d_order])
axes[1, 0].set_ylabel('ACF at lag 6h')
axes[1, 0].set_title('(c) MOMENT regimes vs Persistence \u2014 clear separation')

# (d) FE regimes vs ACF
for idx, e in enumerate(e_order):
    vals = acf6[lab_fe_m == e]
    bp = axes[1, 1].boxplot([vals], positions=[idx], widths=0.6, patch_artist=True)
    bp['boxes'][0].set_facecolor('steelblue'); bp['boxes'][0].set_alpha(0.6)
axes[1, 1].set_xticks(range(K_fe))
axes[1, 1].set_xticklabels([f'E{e}' for e in e_order])
axes[1, 1].set_ylabel('ACF at lag 6h')
axes[1, 1].set_title('(d) FE regimes vs Persistence \u2014 no separation')

for ax in axes.flat:
    ax.grid(alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

## 9. Mean-Reversion Analysis

The residual follows a mean-reverting process: $r_t = (1-\alpha)\, r_{t-1} + s_t$. The speed $\alpha = 1 - \text{ACF}(1)$ determines how fast a shock decays. The half-life is $-\ln 2 / \ln(1-\alpha)$.

Within the same economic regime, $\alpha$ varies enormously depending on the dynamic regime \u2014 a factor of 5\u00d7 in E3 (Baseload).

In [ ]:
# Compute alpha per window
alpha = np.array([1.0 - _acf(wr_mom[i].astype(np.float64), 1) for i in range(N)])

# Per-cell alpha (cells with n >= 20)
cells = []
for e in range(K_fe):
    for d in range(K_mom):
        mask = (lab_fe_m == e) & (lab_mom_m == d)
        n_cell = mask.sum()
        if n_cell >= 20:
            a = alpha[mask].mean()
            hl = -np.log(2) / np.log(1 - a) if a < 1 else np.inf
            cells.append({'E': e, 'D': d, 'n': int(n_cell),
                         'alpha': round(a, 3), 'half_life': round(hl, 1)})

df_cells = pd.DataFrame(cells)
print(f'Cells with n >= 20: {len(df_cells)}')
print()
print(df_cells.to_string(index=False))

In [ ]:
# eta-squared of alpha against each axis and the joint partition
eta2_E = eta2(alpha, lab_fe_m)
eta2_D = eta2(alpha, lab_mom_m)

# Joint eta-squared
joint = lab_fe_m * 100 + lab_mom_m  # composite label
eta2_ED = eta2(alpha, joint)

print(f'\u03b7\u00b2(E) on \u03b1:    {eta2_E:.3f}  \u2014 economic axis explains {100*eta2_E:.1f}% of \u03b1 variance')
print(f'\u03b7\u00b2(D) on \u03b1:    {eta2_D:.3f}  \u2014 dynamic axis explains {100*eta2_D:.1f}% of \u03b1 variance')
print(f'\u03b7\u00b2(E,D) on \u03b1:  {eta2_ED:.3f} \u2014 joint explains {100*eta2_ED:.1f}% of \u03b1 variance')
print(f'\nD governs \u03b1: the persistence axis explains {eta2_D/eta2_E:.1f}\u00d7 more \u03b1 variance than the price axis.')

In [ ]:
# Alpha range within economic regimes
print(f'\u03b1 range within economic regimes (cells with n \u2265 20):')
for e in sorted(df_cells['E'].unique()):
    sub = df_cells[df_cells['E'] == e]
    if len(sub) > 1:
        a_E = alpha[lab_fe_m == e].mean()
        hl_E = -np.log(2) / np.log(1 - a_E)
        print(f'  E{e}: \u03b1(E)={a_E:.3f} (hl={hl_E:.1f}h), '
              f'range \u03b1={sub["alpha"].min():.3f}\u2013{sub["alpha"].max():.3f} '
              f'(hl={sub["half_life"].max():.0f}h\u2013{sub["half_life"].min():.0f}h)')

In [ ]:
# Alpha heatmap in joint (E, D) space
alpha_grid = np.full((K_fe, K_mom), np.nan)
for _, row in df_cells.iterrows():
    alpha_grid[int(row['E']), int(row['D'])] = row['alpha']

# Sort rows/cols by regime mean
e_sort = np.argsort([lmp_mean[lab_fe_m == e].mean() for e in range(K_fe)])
d_sort = np.argsort([acf6[lab_mom_m == d].mean() for d in range(K_mom)])
alpha_sorted = alpha_grid[np.ix_(e_sort, d_sort)]

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(alpha_sorted, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(K_mom))
ax.set_xticklabels([f'D{d}' for d in d_sort])
ax.set_yticks(range(K_fe))
ax.set_yticklabels([f'E{e}' for e in e_sort])
ax.set_xlabel('Dynamic regime (persistence \u2192)')
ax.set_ylabel('Economic regime (price \u2192)')
ax.set_title('Mean-Reversion Speed \u03b1 in Joint (E, D) Space')
plt.colorbar(im, label='\u03b1 (higher = faster reversion)')
for i in range(K_fe):
    for j in range(K_mom):
        v = alpha_sorted[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=7)
plt.tight_layout()
plt.show()

## 10. BIC Model Comparison

We compare two parameterizations:
- **One-axis model** $\alpha(E)$: mean-reversion estimated per economic regime (18 parameters)
- **Two-axis model** $\alpha(E,D)$: mean-reversion estimated per joint cell ($2 \times n_{\text{cells}}$ parameters)

If the dynamic axis carries no information beyond what E already provides, the additional parameters would not justify a better fit.

In [ ]:
# Per-window sigma
sigma = np.array([np.std(np.diff(wr_mom[i].astype(np.float64))) for i in range(N)])


def bic_comparison(values, lab1, lab2, name):
    """Compare one-axis (E only) vs two-axis (E,D) Gaussian models via BIC."""
    # Model 1: values ~ N(mu_E, sigma_E) per E
    ll1, k1 = 0, 0
    for e in np.unique(lab1[lab1 >= 0]):
        v = values[lab1 == e]
        if len(v) > 1:
            mu, sig = v.mean(), v.std()
            if sig > 1e-15:
                ll1 += -0.5 * len(v) * (np.log(2*np.pi*sig**2) + 1)
                k1 += 2

    # Model 2: values ~ N(mu_ED, sigma_ED) per (E,D)
    ll2, k2 = 0, 0
    joint_lab = lab1 * 100 + lab2
    for ed in np.unique(joint_lab[(lab1 >= 0) & (lab2 >= 0)]):
        v = values[joint_lab == ed]
        if len(v) > 1:
            mu, sig = v.mean(), v.std()
            if sig > 1e-15:
                ll2 += -0.5 * len(v) * (np.log(2*np.pi*sig**2) + 1)
                k2 += 2

    n = (lab1 >= 0).sum()
    bic1 = -2*ll1 + k1*np.log(n)
    bic2 = -2*ll2 + k2*np.log(n)
    delta = bic2 - bic1
    print(f'{name}:')
    print(f'  One-axis:  {k1} params, BIC = {bic1:,.0f}')
    print(f'  Two-axis:  {k2} params, BIC = {bic2:,.0f}')
    print(f'  \u0394BIC = {delta:+,.0f} {"\u2190 two-axis wins" if delta < 0 else "\u2190 one-axis wins"}')
    return delta


delta_alpha = bic_comparison(alpha, lab_fe_m, lab_mom_m, 'Mean-reversion \u03b1')
print()
delta_sigma = bic_comparison(sigma, lab_fe_m, lab_mom_m, 'Volatility \u03c3(\u0394r)')

## 11. Two-Axis Grid

The final view: each window plotted in the space of price level \u00d7 persistence. All four quadrants are populated, confirming that the market cannot be described by a single state label.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(lmp_mean, acf6, c=lab_fe_m, cmap='tab10', s=5, alpha=0.4)
ax.set_xlabel('LMP Mean ($/MWh)')
ax.set_ylabel('ACF at lag 6h')
ax.set_title(f'Two-Axis Regime Space (ARI = {ari:.3f})')
ax.axhline(np.median(acf6), color='gray', ls=':', lw=0.8)
ax.axvline(np.median(lmp_mean), color='gray', ls=':', lw=0.8)

# Quadrant labels
ax.text(0.02, 0.98, 'Low price\nHigh persistence', transform=ax.transAxes,
        ha='left', va='top', fontsize=9, color='gray')
ax.text(0.98, 0.98, 'High price\nHigh persistence', transform=ax.transAxes,
        ha='right', va='top', fontsize=9, color='gray')
ax.text(0.02, 0.02, 'Low price\nLow persistence', transform=ax.transAxes,
        ha='left', va='bottom', fontsize=9, color='gray')
ax.text(0.98, 0.02, 'High price\nLow persistence', transform=ax.transAxes,
        ha='right', va='bottom', fontsize=9, color='gray')

ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Summary

The pipeline discovers two quasi-orthogonal axes of regime structure:

| | Economic (FE) | Dynamic (MOMENT) |
|---|---|---|
| **Regimes** | 9 (E0\u2013E8) | 9 (D0\u2013D8) |
| **Separates** | Price level ($30\u2013$151/MWh) | Persistence (ACF 0.38\u20130.94) |
| **\u03b7\u00b2 on target** | 0.495 | 0.420 |
| **Governs \u03b1** | \u03b7\u00b2 = 0.161 | \u03b7\u00b2 = 0.376 |

**ARI = 0.012** \u2014 the axes are quasi-orthogonal.

**\u0394BIC = \u22123,875** \u2014 the two-axis model is overwhelmingly preferred.

Within the same price regime (E3, Baseload), mean-reversion half-life varies from 4h to 22h depending on the dynamic regime \u2014 a 5\u00d7 factor invisible to single-axis models.